In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from pathlib import Path
import ot
import scipy as sp
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from utils_mocap import LoadCloudPoint, DistanceProfile
from utils_mocap import compute_W_matrix_distance_matrix_input
from utils_mocap import plot_3d_points_and_connections

In [ ]:
import random

random.seed(10)

lcp = LoadCloudPoint(filepath="datasets/0005_Jogging001.csv")
source_pc, target_pc = lcp.get_two_random_point_cloud()

dp = DistanceProfile(source_pc, target_pc)
distance_matrix = dp.compute_L2_matrix()

# Find KNN matrix

In [ ]:

import numpy as np
from sklearn.neighbors import NearestNeighbors
import plotly.graph_objects as go

def plot_kth_neighbor_graph(points, k):
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(points)
    distances, indices = nbrs.kneighbors(points)
    indices = indices[:,1:]

    # Build edge lines
    edge_x, edge_y, edge_z = [], [], []
    for i in range(points.shape[0]):
        for j in indices[i]:
            p1 = points[i]
            p2 = points[j]
            edge_x += [p1[0], p2[0], None]
            edge_y += [p1[1], p2[1], None]
            edge_z += [p1[2], p2[2], None]

    # Build figure
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode="lines",
        line=dict(width=2),
        hoverinfo="none"
    ))

    fig.add_trace(go.Scatter3d(
        x=points[:,0],
        y=points[:,1],
        z=points[:,2],
        mode="markers",
        marker=dict(size=4),
        hoverinfo="none"
    ))

    fig.update_layout(
        title="Interactive 3D kNN Mocap Graph",
        scene=dict(aspectmode="data"),
        width=900,
        height=700
    )

    fig.show()

    # return knn adjency matrix
    knn_adj_matrix = np.zeros((points.shape[0], points.shape[0]))
    for i in range(points.shape[0]):
        for j in indices[i]:
            knn_adj_matrix[i, j] = 1

    # make an adjency matrix with each element the distance using euclidean distance
    # if there is no edge, set it to np.inf
    knn_weighted_adj_matrix = np.zeros((points.shape[0], points.shape[0]))

    for i in range(points.shape[0]):
        for idx, j in enumerate(indices[i]):

            # take floor of distance to avoid very small float issues
            knn_weighted_adj_matrix[i, j] = np.floor(distances[i][idx])


    # make an adjency matrix with each element is the k - i where k in the kNN and i is the index of the neighbor
    knn_weighted_naive_matrix = np.zeros((points.shape[0], points.shape[0]))

    for i in range(points.shape[0]):
        for idx, j in enumerate(indices[i]):
            knn_weighted_naive_matrix[i, j] = k - idx

    return knn_adj_matrix, knn_weighted_adj_matrix, knn_weighted_naive_matrix


(src_adj_matrix, src_weighted_adj_matrix, src_weighted_naive_matrix) = plot_kth_neighbor_graph(source_pc, k=5)


# weighted adj matrix

In [ ]:
src_weighted_adj_matrix

In [ ]:
src_weighted_naive_matrix

# Repeat for target

In [ ]:
(tar_adj_matrix, tar_weighted_adj_matrix, tar_weighted_naive_matrix) = plot_kth_neighbor_graph(target_pc, k=5)

# Make interactions graph

In [ ]:
# package into a data structure with {'src_index': ..., 'tar_index': ..., 'src_interactions': ..., 'tar_interactions': ...}

data_structure = {}

# list all indices of src points with {0.0: 0, 1.0: 1, 2.0: 2, ...}
data_structure['src_index'] = {float(i): i for i in range(source_pc.shape[0])}
# list all indices of tar points with {0.0: 0, 1.0: 1, 2.0: 2, ...}
data_structure['tar_index'] = {float(i): i for i in range(target_pc.shape[0])}

# list all interations for src points wit form [[0, np.int32(5)], [1, np.int32(3)], ...] with one listing for each weight i.e. a weight of 3 will have 3 listings
data_structure['src_interactions'] = []
for i in range(src_weighted_adj_matrix.shape[0]):
    for j in range(src_weighted_adj_matrix.shape[1]):
        weight = int(src_weighted_adj_matrix[i, j])
        for _ in range(weight):
            data_structure['src_interactions'].append([i, np.int32(j)])

# list all interations for tar points wit form [[0, np.int32(5)], [1, np.int32(3)], ...] with one listing for each weight i.e. a weight of 3 will have 3 listings
data_structure['tar_interactions'] = []
for i in range(tar_weighted_adj_matrix.shape[0]):
    for j in range(tar_weighted_adj_matrix.shape[1]):
        weight = int(tar_weighted_adj_matrix[i, j])
        for _ in range(weight):
            data_structure['tar_interactions'].append([i, np.int32(j)])

data_structure

In [ ]:
data_structure.keys()

# Shove in model

In [ ]:
import dev.util as util
from dev.util import logger
import matplotlib.pyplot as plt
from model.GromovWassersteinLearning import GromovWassersteinLearning
from model.BAPG import process_interaction_data
import numpy as np
import pickle
import torch.optim as optim
from torch.optim import lr_scheduler
import time

In [ ]:
time_GWEMBED = {}
time_BAPG = {}

node_accuracy_GWEMBED = {}
node_accuracy_BAPG = {}

nn = 'mc3'
n = 'test'
i = 0

n_nodes = ['test']
n_noises = 1

for n in n_nodes:
    for i in range(n_noises):
        time_GWEMBED[(n, i)] = []
        time_BAPG[(n, i)] = []
        node_accuracy_BAPG[(n, i)] = []
        node_accuracy_GWEMBED[(n, i)] = []

data_name = 'syn_{}_{}_{}'.format(nn, n, i)
result_folder = 'match_syn'
cost_type = ['cosine']
method = ['proximal']

util.makedirs(result_folder)

data_mc3 = data_structure


print(len(data_mc3['src_index']))
print(len(data_mc3['tar_index']))
print(len(data_mc3['src_interactions']))
print(len(data_mc3['tar_interactions']))

connects = np.zeros((len(data_mc3['src_index']), len(data_mc3['src_index'])))
for item in data_mc3['src_interactions']:
    connects[item[0], item[1]] += 1
plt.imshow(connects)
plt.savefig('{}/{}_src.png'.format(result_folder, data_name))
plt.close('all')

connects = np.zeros((len(data_mc3['tar_index']), len(data_mc3['tar_index'])))
for item in data_mc3['tar_interactions']:
    connects[item[0], item[1]] += 1
plt.imshow(connects)
plt.savefig('{}/{}_tar.png'.format(result_folder, data_name))
plt.close('all')

opt_dict = {'epochs': 5,
            'batch_size': 10000,
            'use_cuda': False,
            'strategy': 'soft',
            'beta': 1e-1,
            'outer_iteration': 400,
            'inner_iteration': 1,
            'sgd_iteration': 300,
            'prior': False,
            'prefix': result_folder,
            'display': True}

for m in method:
    for c in cost_type:
        hyperpara_dict = {'src_number': len(data_mc3['src_index']),
                          'tar_number': len(data_mc3['tar_index']),
                          'dimension': 20,
                          'loss_type': 'L2',
                          'cost_type': c,
                          'ot_method': m}

        gwd_model = GromovWassersteinLearning(hyperpara_dict)

        # initialize optimizer
        optimizer = optim.Adam(gwd_model.gwl_model.parameters(), lr=1e-3)
        scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=0.8)

        print("\nRunning Gromov-Wasserstein learning {}".format(data_name))

        # Gromov-Wasserstein learning
        time_start = time.time()
        gwd_model.train_without_prior(data_mc3, optimizer, opt_dict, scheduler=None)
        time_end = time.time()
        node_accuracy_GWEMBED[(n, i)].append(gwd_model.NC1)
        time_GWEMBED[(n, i)].append(time_end - time_start)
        print('Gromov-Wasserstein learning time cost: {:.4f}s'.format(time_end - time_start))

In [ ]:
# package into a data structure with {'src_index': ..., 'tar_index': ..., 'src_interactions': ..., 'tar_interactions': ...}

data_structure = {}

# list all indices of src points with {0.0: 0, 1.0: 1, 2.0: 2, ...}
data_structure['src_index'] = {float(i): i for i in range(source_pc.shape[0])}
# list all indices of tar points with {0.0: 0, 1.0: 1, 2.0: 2, ...}
data_structure['tar_index'] = {float(i): i for i in range(target_pc.shape[0])}

# list all interations for src points wit form [[0, np.int32(5)], [1, np.int32(3)], ...] with one listing for each weight i.e. a weight of 3 will have 3 listings
data_structure['src_interactions'] = []
for i in range(src_weighted_naive_matrix.shape[0]):
    for j in range(src_weighted_naive_matrix.shape[1]):
        weight = int(src_weighted_naive_matrix[i, j])
        for _ in range(weight):
            data_structure['src_interactions'].append([i, np.int32(j)])

# list all interations for tar points wit form [[0, np.int32(5)], [1, np.int32(3)], ...] with one listing for each weight i.e. a weight of 3 will have 3 listings
data_structure['tar_interactions'] = []
for i in range(tar_weighted_naive_matrix.shape[0]):
    for j in range(tar_weighted_naive_matrix.shape[1]):
        weight = int(tar_weighted_naive_matrix[i, j])
        for _ in range(weight):
            data_structure['tar_interactions'].append([i, np.int32(j)])

data_structure


In [29]:
time_GWEMBED = {}
time_BAPG = {}

node_accuracy_GWEMBED = {}
node_accuracy_BAPG = {}

nn = 'mc3'
n = 'test'
i = 0

n_nodes = ['test']
n_noises = 1

for n in n_nodes:
    for i in range(n_noises):
        time_GWEMBED[(n, i)] = []
        time_BAPG[(n, i)] = []
        node_accuracy_BAPG[(n, i)] = []
        node_accuracy_GWEMBED[(n, i)] = []

data_name = 'syn_{}_{}_{}'.format(nn, n, i)
result_folder = 'match_syn'
cost_type = ['cosine']
method = ['proximal']

util.makedirs(result_folder)

data_mc3 = data_structure


print(len(data_mc3['src_index']))
print(len(data_mc3['tar_index']))
print(len(data_mc3['src_interactions']))
print(len(data_mc3['tar_interactions']))

connects = np.zeros((len(data_mc3['src_index']), len(data_mc3['src_index'])))
for item in data_mc3['src_interactions']:
    connects[item[0], item[1]] += 1
plt.imshow(connects)
plt.savefig('{}/{}_src.png'.format(result_folder, data_name))
plt.close('all')

connects = np.zeros((len(data_mc3['tar_index']), len(data_mc3['tar_index'])))
for item in data_mc3['tar_interactions']:
    connects[item[0], item[1]] += 1
plt.imshow(connects)
plt.savefig('{}/{}_tar.png'.format(result_folder, data_name))
plt.close('all')

opt_dict = {'epochs': 5,
            'batch_size': 10000,
            'use_cuda': False,
            'strategy': 'soft',
            'beta': 1e-1,
            'outer_iteration': 400,
            'inner_iteration': 1,
            'sgd_iteration': 300,
            'prior': False,
            'prefix': result_folder,
            'display': True}

for m in method:
    for c in cost_type:
        hyperpara_dict = {'src_number': len(data_mc3['src_index']),
                          'tar_number': len(data_mc3['tar_index']),
                          'dimension': 20,
                          'loss_type': 'L2',
                          'cost_type': c,
                          'ot_method': m}

        gwd_model = GromovWassersteinLearning(hyperpara_dict)

        # initialize optimizer
        optimizer = optim.Adam(gwd_model.gwl_model.parameters(), lr=1e-3)
        scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=0.8)

        print("\nRunning Gromov-Wasserstein learning {}".format(data_name))

        # Gromov-Wasserstein learning
        time_start = time.time()
        gwd_model.train_without_prior(data_mc3, optimizer, opt_dict, scheduler=None)
        time_end = time.time()
        node_accuracy_GWEMBED[(n, i)].append(gwd_model.NC1)
        time_GWEMBED[(n, i)].append(time_end - time_start)
        print('Gromov-Wasserstein learning time cost: {:.4f}s'.format(time_end - time_start))

53
53
795
795

Running Gromov-Wasserstein learning syn_mc3_test_0
sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/300: loss=113.752464.
inner 10/300: loss=111.550575.
inner 20/300: loss=108.534058.
inner 30/300: loss=104.722656.
inner 40/300: loss=100.254868.
inner 50/300: loss=95.358398.
inner 60/300: loss=90.276566.
inner 70/300: loss=85.231949.
inner 80/300: loss=80.432877.
inner 90/300: loss=76.053963.
inner 100/300: loss=72.204231.
inner 110/300: loss=68.917145.
inner 120/300: loss=66.168335.
inner 130/300: loss=63.907913.
inner 140/300: loss=62.079918.
inner 150/300: loss=60.621437.
inner 160/300: loss=59.462078.
inner 170/300: loss=58.534050.
inner 180/300: loss=57.783249.
inner 190/300: loss=57.170887.
inner 200/300: loss=56.669079.
inner 210/300: loss=56.256573.
inner 220/300: loss=55.916454.
inner 230/300: loss=55.635063.
inner 240/300: loss=55.401302.
inner 250/300: loss=55.206108.


INFO:dev.util:Train Epoch: 0
INFO:dev.util:- node correctness: 60.3774%, 64.1509%
INFO:dev.util:- edge correctness: 75.0943%, 79.6226%
INFO:dev.util:- GW distance = 0.0171.


inner 260/300: loss=55.042130.
inner 270/300: loss=54.903408.
inner 280/300: loss=54.785130.
inner 290/300: loss=54.683411.
sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/100: loss=40.026939.
inner 10/100: loss=39.803974.
inner 20/100: loss=39.633446.
inner 30/100: loss=39.567741.
inner 40/100: loss=39.526936.
inner 50/100: loss=39.491421.
inner 60/100: loss=39.460720.
inner 70/100: loss=39.433784.
inner 80/100: loss=39.409340.
inner 90/100: loss=39.386658.


INFO:dev.util:Train Epoch: 1
INFO:dev.util:- node correctness: 62.2642%, 62.2642%
INFO:dev.util:- edge correctness: 78.8679%, 81.1321%
INFO:dev.util:- GW distance = 0.0108.


sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/100: loss=34.785156.
inner 10/100: loss=34.621250.
inner 20/100: loss=34.508751.
inner 30/100: loss=34.476875.
inner 40/100: loss=34.457191.
inner 50/100: loss=34.438595.
inner 60/100: loss=34.422375.
inner 70/100: loss=34.407780.
inner 80/100: loss=34.394009.


...

inner 90/100: loss=34.380806.
sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/100: loss=34.794605.
inner 10/100: loss=34.695412.
inner 20/100: loss=34.630123.
inner 30/100: loss=34.610153.
inner 40/100: loss=34.595184.
inner 50/100: loss=34.581139.
inner 60/100: loss=34.568851.
inner 70/100: loss=34.557564.
inner 80/100: loss=34.546825.
inner 90/100: loss=34.536488.


INFO:dev.util:Train Epoch: 3
INFO:dev.util:- node correctness: 60.3774%, 66.0377%
INFO:dev.util:- edge correctness: 76.6038%, 81.1321%
INFO:dev.util:- GW distance = 0.0027.


sinkhorn iter 0/400
sinkhorn iter 100/400
sinkhorn iter 200/400
sinkhorn iter 300/400
inner 0/100: loss=36.382744.
inner 10/100: loss=36.331818.
inner 20/100: loss=36.294567.
inner 30/100: loss=36.275391.
inner 40/100: loss=36.258614.
inner 50/100: loss=36.243332.
inner 60/100: loss=36.229588.
inner 70/100: loss=36.216812.
inner 80/100: loss=36.204685.


INFO:dev.util:Train Epoch: 4
INFO:dev.util:- node correctness: 60.3774%, 66.0377%
INFO:dev.util:- edge correctness: 75.8491%, 79.6226%
INFO:dev.util:- GW distance = 0.0007.


inner 90/100: loss=36.193069.
Gromov-Wasserstein learning time cost: 5.0039s
